# `filter` — Comprehensive Reference

`filter` filters a DataFrame using intuitive CLI-style string expressions or programmatic dictionaries.

Under the hood, `filter` compiles conditions into high-performance Polars expressions evaluated directly in native C++/Rust memory.

### Calling Styles

`filter` supports two clean, unambiguous calling styles:

- **String expressions** (matches CLI `-filter` syntax; handles spaces directly):
  `df.filter("[bill length mm] > 40 and species == 'Adelie'")`
- **Dictionaries** (ideal for dynamic programmatic keys and Option 1 / Option 2 ranges):
  `df.filter({"bill length mm": "> 40", "species": "Adelie"})`
  `df.filter({"body_mass_g": ("between", 3500, 4000)})`
  `df.filter({"body_mass_g": ((">=", 3500), ("<=", 4000))})`
- **Top-level library verb**:
  `pt.filter(df, "species == 'Adelie' and body_mass_g > 5000")`

> [!IMPORTANT]
> **Why `pytae` Strictly Enforces `==` and Disallows Kwargs**:
> In Python syntax, keyword arguments like `df.filter(species="Adelie")` force the use of a single `=`. In `pytae`, single `=` is reserved exclusively for column assignments in `-mutate`. To prevent subtle bugs and ensure **"one and only one obvious way"**, `filter()` strictly requires `==` and raises an immediate `TypeError` if keyword arguments are passed.

---

### Comprehensive Reference Table

| Category | Example Syntax | Meaning / Target |
|---|---|---|
| **Equality** | `"species == 'Adelie'"` or `{"species": "Adelie"}` | Exact equality match (`==` strictly enforced; single `=` raises error) |
| **Inequality** | `"species != 'Adelie'"` or `{"species": ("!=", "Adelie")}` | Inequality match |
| **Relational Operators** | `"body_mass_g > 5000"` or `{"body_mass_g": (">", 5000)}` | `>`, `>=`, `<`, `<=`, `==`, `!=` |
| **Set Membership** | `"species in ['Adelie', 'Gentoo']"` or `{"species": ['Adelie', 'Gentoo']}` | In candidate set / list |
| **Exclusion** | `"sex not in ['Male']"` or `{"sex": ("not in", ['Male'])}` | Not in candidate set |
| **Boolean AND** | `"species == 'Gentoo' and body_mass_g > 4500"` | Logical conjunction (comma `,` and bitwise `&` disallowed) |
| **Boolean OR** | `"island == 'Dream' or island == 'Torgersen'"` | Logical disjunction (bitwise `\|` disallowed) |
| **Grouping / Precedence** | `"(species == 'Adelie' or species == 'Chinstrap') and mass > 4000"` | Arbitrary nested boolean logic with parentheses |
| **Range Option 1 (`between`)** | `{"body_mass_g": ("between", 3500, 4000)}` | Inclusive range (`min <= x <= max`) via Polars `.is_between()` |
| **Range Option 2 (operator tuples)** | `{"body_mass_g": ((">=", 3500), ("<=", 4000))}` | Multiple conditions on same column via clean `(op, val)` pairs |
| **String Range Expression** | `"body_mass_g >= 3500 and body_mass_g <= 4000"` | Explicit boolean bounds in string expression |
| **Columns with Spaces** | `"[bill length mm] > 40"` or `{"bill length mm": (">", 40)}` | Spaced column syntax (brackets in strings, plain keys in dicts) |
| **String Prefixes/Suffixes** | `{"species": ("startswith", "Ad")}` or `{"species": ("endswith", "strap")}` | String prefix / suffix search |
| **Multiple Prefixes/Suffixes** | `{"species": ("startswith", ["Ad", "Ge"])}` | Matches any prefix / suffix in sequence |
| **String Substring / Regex** | `{"species": ("contains", "adel")}` or `{"species": ("regex", r"oo")}` | Substring match or regex search |
| **Null & NaN Checks** | `"sex.is_not_null()"`, `{"sex": "is_null()"}`, `"score.is_nan()"`, `{"score": "is_nan()"}` | Missing data (`null`) and IEEE 754 float NaNs (`NaN`) via universal method tokens (expressions & dicts) |
| **Column-to-Column** | `"exposure > limit"` or `"tranche_id == deal_id"` | Compare two columns using bare column names |
| **Variable Injection** | `"exposure > @threshold and tranche == col(@target_col)"` | External scalar (`@var`) and column (`col(@var)`) references |

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
tips = pt.sample_data['tips']
titanic = pt.sample_data['titanic']
print("Loaded sample datasets:", list(pt.sample_data.keys()))

Loaded sample datasets: ['anagrams', 'anscombe', 'attention', 'brain_networks', 'car_crashes', 'diamonds', 'dots', 'dowjones', 'exercise', 'flights', 'fmri', 'geyser', 'glue', 'healthexp', 'iris', 'mpg', 'penguins', 'planets', 'seaice', 'taxis', 'tips', 'titanic']


## 1. Primary Calling Styles: String Expressions and Dictionaries

`filter()` accepts conditions as string expressions matching the CLI `-filter` syntax, or plain dictionaries for programmatic filters. Both direct method chaining (`df.filter(...)`) and the top-level library verb (`pt.filter(df, ...)`) are supported.

In [2]:
# 1. String expressions — clean, readable, matches pytae CLI -filter syntax
# Uses '==' for equality and 'and' for logical conjunction
(
    penguins
    .filter("species == 'Adelie' and island == 'Dream'")
    .head(5)
)

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Dream""",39.5,16.7,178.0,3250.0,"""Female"""
"""Adelie""","""Dream""",37.2,18.1,178.0,3900.0,"""Male"""
"""Adelie""","""Dream""",39.5,17.8,188.0,3300.0,"""Female"""
"""Adelie""","""Dream""",40.9,18.9,184.0,3900.0,"""Male"""
"""Adelie""","""Dream""",36.4,17.0,195.0,3325.0,"""Female"""


In [3]:
# 2. Plain dictionaries — ideal for programmatic pipelines or dynamic key mapping
(
    penguins
    .filter({"species": "Adelie", "island": "Dream"})
    .head(5)
)

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Dream""",39.5,16.7,178.0,3250.0,"""Female"""
"""Adelie""","""Dream""",37.2,18.1,178.0,3900.0,"""Male"""
"""Adelie""","""Dream""",39.5,17.8,188.0,3300.0,"""Female"""
"""Adelie""","""Dream""",40.9,18.9,184.0,3900.0,"""Male"""
"""Adelie""","""Dream""",36.4,17.0,195.0,3325.0,"""Female"""


In [4]:
# 3. Composing multiple conditions — either chained or passed as multiple positional arguments
(
    penguins
    .filter("body_mass_g > 4000", "island == 'Biscoe'")
    .head(5)
)

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Biscoe""",40.1,18.9,188.0,4300.0,"""Male"""
"""Adelie""","""Biscoe""",42.0,19.5,200.0,4050.0,"""Male"""
"""Adelie""","""Biscoe""",41.3,21.1,195.0,4400.0,"""Male"""
"""Adelie""","""Biscoe""",41.1,18.2,192.0,4050.0,"""Male"""
"""Adelie""","""Biscoe""",41.1,19.1,188.0,4100.0,"""Male"""


In [5]:
# 4. Top-level function pt.filter(df, ...) — works identically to DataFrame method
pt.filter(penguins, "species == 'Gentoo' and body_mass_g > 5000").head(5)

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Gentoo""","""Biscoe""",50.0,16.3,230.0,5700.0,"""Male"""
"""Gentoo""","""Biscoe""",50.0,15.2,218.0,5700.0,"""Male"""
"""Gentoo""","""Biscoe""",47.6,14.5,215.0,5400.0,"""Male"""
"""Gentoo""","""Biscoe""",46.7,15.3,219.0,5200.0,"""Male"""
"""Gentoo""","""Biscoe""",46.8,15.4,215.0,5150.0,"""Male"""


## 2. Relational Comparisons (`==`, `!=`, `<`, `>`, `<=`, `>=`)

Comparison filters can be written directly in string expressions or as operator tuples inside dictionaries:
- String expressions: `"body_mass_g > 5000"`, `"species == 'Gentoo'"`
- Dictionary with operator tuples: `{"body_mass_g": (">", 5000)}`, `{"island": ("!=", "Dream")}`

> [!IMPORTANT]
> **Strict Equality Enforcement**: In `pytae`, equality comparisons in filter strings strictly require `==`. Single `=` is reserved exclusively for column assignments in `-mutate`. Using single `=` in filter expressions raises an immediate diagnostic error.

In [6]:
# Numeric comparisons via string expressions
(
    penguins
    .filter("body_mass_g > 5000 and flipper_length_mm >= 220")
    .select('species', 'island', 'body_mass_g', 'flipper_length_mm')
    .head(5)
)

species,island,body_mass_g,flipper_length_mm
str,str,f64,f64
"""Gentoo""","""Biscoe""",5700.0,230.0
"""Gentoo""","""Biscoe""",6300.0,221.0
"""Gentoo""","""Biscoe""",5350.0,222.0
"""Gentoo""","""Biscoe""",5550.0,220.0
"""Gentoo""","""Biscoe""",5250.0,222.0


In [7]:
# Dictionary with operator tuples: (op, value)
(
    penguins
    .filter({"body_mass_g": (">", 5500), "island": ("!=", "Dream")})
    .select('species', 'island', 'body_mass_g')
    .head(5)
)

species,island,body_mass_g
str,str,f64
"""Gentoo""","""Biscoe""",5700.0
"""Gentoo""","""Biscoe""",5700.0
"""Gentoo""","""Biscoe""",5550.0
"""Gentoo""","""Biscoe""",5850.0
"""Gentoo""","""Biscoe""",5850.0


In [8]:
# Multiple comparisons in string expressions
(
    penguins
    .filter("body_mass_g <= 3000 and bill_length_mm < 37")
    .select('species', 'island', 'bill_length_mm', 'body_mass_g')
    .head(5)
)

species,island,bill_length_mm,body_mass_g
str,str,f64,f64
"""Adelie""","""Biscoe""",34.5,2900.0
"""Adelie""","""Biscoe""",36.5,2850.0
"""Adelie""","""Biscoe""",36.4,2850.0
"""Adelie""","""Dream""",33.1,2900.0


## 3. Set Membership (`in` and `not in`)

To match against a list of candidate values:
- Use SQL/Python-style `in` or `not in` in string expressions: `"species in ['Adelie', 'Chinstrap']"`
- Pass a candidate list or exclusion tuple in a dictionary: `{"species": ['Adelie', 'Chinstrap']}`, `{"sex": ("not in", ['Male'])}`

In [9]:
# Set membership via string expression
(
    penguins
    .filter("species in ['Adelie', 'Chinstrap'] and island in ['Dream', 'Biscoe']")
    .by('species', 'island')
    .agg("n = count, mean_mass = body_mass_g:mean")
)

species,island,n,mean_mass
str,str,i64,f64
"""Adelie""","""Biscoe""",44,3709.659091
"""Adelie""","""Dream""",56,3688.392857
"""Chinstrap""","""Dream""",68,3733.088235


In [10]:
# 'in' and 'not in' inside string expressions
(
    penguins
    .filter("species in ['Adelie', 'Gentoo'] and island not in ['Torgersen']")
    .by('species', 'island')
    .agg("n = count")
)

species,island,n
str,str,i64
"""Adelie""","""Biscoe""",44
"""Adelie""","""Dream""",56
"""Gentoo""","""Biscoe""",124


In [11]:
# Membership and exclusion using dictionary syntax
(
    penguins
    .filter({
        "species": ("not in", ['Gentoo']),
        "island": ['Dream', 'Torgersen'],
    })
    .select('species', 'island', 'sex', 'body_mass_g')
    .head(5)
)

species,island,sex,body_mass_g
str,str,str,f64
"""Adelie""","""Torgersen""","""Male""",3750.0
"""Adelie""","""Torgersen""","""Female""",3800.0
"""Adelie""","""Torgersen""","""Female""",3250.0
"""Adelie""","""Torgersen""",null,null
"""Adelie""","""Torgersen""","""Female""",3450.0


## 4. Boolean Logic: `and`, `or`, and Parentheses `(...)`

`pytae` supports rich boolean expressions in filter strings with:
- `and` for logical conjunction
- `or` for logical disjunction
- Arbitrary nesting with parentheses `(...)` for explicit precedence control

> [!NOTE]
> Following the Zen of Python (*"There should be one-- and preferably only one --obvious way to do it"*), `pytae` strictly uses `and` and `or`. Bitwise `&` and `|` are disallowed, and commas `,` are strictly punctuation for function arguments and lists, not boolean operators.

In [12]:
# Logical conjunction with 'and'
(
    penguins
    .filter("species == 'Gentoo' and body_mass_g > 5500 and sex == 'Female'")
    .select('species', 'island', 'sex', 'body_mass_g')
    .head(5)
)

species,island,sex,body_mass_g
str,str,str,f64


In [13]:
# Logical disjunction with 'or'
(
    penguins
    .filter("island == 'Dream' or island == 'Torgersen'")
    .by('island')
    .agg("n = count")
)

island,n
str,i64
"""Dream""",124
"""Torgersen""",52


In [14]:
# Combining 'or' and 'and' with parentheses for precedence:
# Keep Adelie or Chinstrap penguins that weigh over 4000g:
(
    penguins
    .filter("(species == 'Adelie' or species == 'Chinstrap') and body_mass_g > 4000")
    .select('species', 'island', 'sex', 'body_mass_g')
    .head(6)
)

species,island,sex,body_mass_g
str,str,str,f64
"""Adelie""","""Torgersen""","""Male""",4675.0
"""Adelie""","""Torgersen""",null,4250.0
"""Adelie""","""Torgersen""","""Male""",4400.0
"""Adelie""","""Torgersen""","""Male""",4500.0
"""Adelie""","""Torgersen""","""Male""",4200.0
"""Adelie""","""Dream""","""Male""",4150.0


In [15]:
# Complex nested logic:
# (Gentoo on Biscoe) OR (Adelie on Dream), with body mass >= 3500
(
    penguins
    .filter("((species == 'Gentoo' and island == 'Biscoe') or (species == 'Adelie' and island == 'Dream')) and body_mass_g >= 3500")
    .by('species', 'island')
    .agg("n = count, mean_mass = body_mass_g:mean")
)

species,island,n,mean_mass
str,str,i64,f64
"""Adelie""","""Dream""",33,3984.090909
"""Gentoo""","""Biscoe""",123,5076.01626


## 5. Range & Interval Filtering: Option 1 and Option 2

Previous versions of data tools frequently used mathematical interval strings like `'[3000, 4000]'` or `'(3000, 4000)'`. In modern `pytae`, string bracket notation was dropped to eliminate ambiguity with Python list literals and ensure unambiguous typing.

Instead, `pytae` provides two clean, powerful options for range filtering:

### Option 1: `("between", min, max)`
Pass `("between", min, max)` inside a dictionary to map directly to Polars' highly optimized native `.is_between(lower, upper)` method:
```python
df.filter({"body_mass_g": ("between", 3000, 4000)})
```

### Option 2: Multi-condition Operator Tuples `(('op1', val1), ('op2', val2))`
Pass a tuple of clean `(operator, value)` pairs inside a dictionary to apply multiple conditions on the same column without string-mashed syntax:
```python
# Inclusive bounds:
df.filter({"body_mass_g": ((">=", 3000), ("<=", 4000))})

# Exclusive or asymmetric bounds:
df.filter({"body_mass_g": ((">", 3000), ("<", 4500))})
df.filter({"bill_length_mm": ((">", 40), ("<", 45))})
```

### String Expressions with `and`
You can also write ranges naturally in filter strings:
```python
df.filter("body_mass_g >= 3000 and body_mass_g <= 4000")
```

In [16]:
# Option 1: ("between", min, max) — inclusive range via dictionary
display(
    penguins
    .filter({"body_mass_g": ("between", 3500, 3800)})
    .select('species', 'island', 'body_mass_g')
    .head(5)
)

species,island,body_mass_g
str,str,f64
"""Adelie""","""Torgersen""",3750.0
"""Adelie""","""Torgersen""",3800.0
"""Adelie""","""Torgersen""",3650.0
"""Adelie""","""Torgersen""",3625.0
"""Adelie""","""Torgersen""",3700.0


In [17]:
# Option 2: Multi-condition operator tuples (('op', val), ...)
# Clean separation of operators and values; perfect for exclusive or asymmetric bounds:
(
    penguins
    .filter({
        "bill_length_mm": ((">", 40), ("<", 45)),
        "body_mass_g": ((">=", 3500), ("<=", 4000)),
    })
    .select('species', 'bill_length_mm', 'body_mass_g')
    .head(5)
)

species,bill_length_mm,body_mass_g
str,f64,f64
"""Adelie""",40.6,3550.0
"""Adelie""",40.5,3950.0
"""Adelie""",40.9,3900.0
"""Adelie""",42.2,3550.0
"""Adelie""",40.8,3900.0


In [18]:
# Standard string expression range using 'and'
(
    penguins
    .filter("body_mass_g >= 3500 and body_mass_g <= 4000 and bill_depth_mm > 18")
    .select('species', 'bill_depth_mm', 'body_mass_g')
    .head(5)
)

species,bill_depth_mm,body_mass_g
str,f64,f64
"""Adelie""",18.7,3750.0
"""Adelie""",20.6,3650.0
"""Adelie""",21.2,3800.0
"""Adelie""",18.7,3600.0
"""Adelie""",19.2,3800.0


## 6. Columns with Spaces & Bracket Notation `[col]`

When column names contain spaces or special punctuation (such as `bill length mm` or `counterparty_class_(irb)`):
- In **string expressions**, wrap the column name in square brackets: `[bill length mm] > 40`
- In **dictionaries**, use the column name directly as the key without any brackets: `{"bill length mm": "> 40"}`

In [19]:
# Create a DataFrame with spaces in column names
penguins_spaces = penguins.rename(columns={
    'bill_length_mm': 'bill length mm',
    'body_mass_g': 'body mass g',
})
print("Columns with spaces:", penguins_spaces.columns.tolist()[:4])

Columns with spaces: ['species', 'island', 'bill length mm', 'bill_depth_mm']


In [20]:
# Filtering spaced columns using bracket notation in string expressions:
(
    penguins_spaces
    .filter("[bill length mm] > 45 and [body mass g] > 4000")
    .select('species', '[bill length mm]', '[body mass g]')
    .head(5)
)

species,bill length mm,body mass g
str,f64,f64
"""Adelie""",46.0,4200.0
"""Adelie""",45.8,4150.0
"""Adelie""",45.6,4600.0
"""Chinstrap""",46.0,4150.0
"""Chinstrap""",52.0,4050.0


In [21]:
# Filtering spaced columns using plain dictionaries (Option 1 & Option 2 supported!):
(
    penguins_spaces
    .filter({
        "bill length mm": (">", 45),
        "body mass g": ("between", 4000, 5000),
    })
    .select('species', '[bill length mm]', '[body mass g]')
    .head(5)
)

species,bill length mm,body mass g
str,f64,f64
"""Adelie""",46.0,4200.0
"""Adelie""",45.8,4150.0
"""Adelie""",45.6,4600.0
"""Chinstrap""",46.0,4150.0
"""Chinstrap""",52.0,4050.0


## 7. String Matching (`startswith`, `endswith`, `contains`, `regex`)

For text/string columns, `pytae` provides high-speed pattern matching inside dictionaries:
- `{"col": ('startswith', prefix)}`: Matches string prefix
- `{"col": ('startswith', [prefix1, prefix2, ...])}`: Matches any of several prefixes
- `{"col": ('endswith', suffix)}` / `{"col": ('endswith', [suffix1, suffix2, ...])}`: Matches suffixes
- `{"col": ('contains', substr)}`: Substring search
- `{"col": ('regex', pattern)}`: Regular expression pattern search

In [22]:
# Prefix and suffix matches via dictionary
(
    penguins
    .filter({
        "species": ('startswith', 'Ad'),
        "island": ('endswith', 'e'),
    })
    .select('species', 'island', 'body_mass_g')
    .head(5)
)

species,island,body_mass_g
str,str,f64
"""Adelie""","""Biscoe""",3400.0
"""Adelie""","""Biscoe""",3600.0
"""Adelie""","""Biscoe""",3800.0
"""Adelie""","""Biscoe""",3950.0
"""Adelie""","""Biscoe""",3800.0


In [23]:
# Multiple prefixes: matches species starting with 'Ad' OR 'Ge'
(
    penguins
    .filter({"species": ('startswith', ['Ad', 'Ge'])})
    .by('species')
    .agg("n = count")
)

species,n
str,i64
"""Adelie""",152
"""Gentoo""",124


In [24]:
# Regex pattern matching (e.g. repeated 'o')
(
    penguins
    .filter({"species": ('regex', r'oo')})
    .by('species')
    .agg("n = count")
)

species,n
str,i64
"""Gentoo""",124


## 8. Missing and Null Checks (`is_null()`, `is_not_null()`, `is_nan()`, `is_not_nan()`)

In Pytae and pure Polars, missing data (`null`) and floating-point non-finite values (`NaN`) follow a unified **method-token standard** across both **string expressions** and **dictionaries**:

- **Missing values (`null`)**: Represent absent / unrecorded values across any data type (string, numeric, date). Handled via `.is_null()` and `.is_not_null()`.
- **Floating-point non-finites (`NaN`)**: Represent IEEE 754 arithmetic results on float columns (`f32`, `f64`). Handled via `.is_nan()` and `.is_not_nan()`. Polars strictly separates `null` from `NaN`.

The exact same method names—`is_null()`, `is_not_null()`, `is_nan()`, and `is_not_nan()`—are used consistently everywhere.

In [25]:
# 1. Missing data checks via string expressions (non-dict):
print("--- Missing sex using string expression (sex.is_null()) ---")
display(
    penguins
    .filter("sex.is_null()")
    .select('species', 'island', 'body_mass_g', 'sex')
    .head(5)
)

print("\n--- Known sex using string expression (sex.is_not_null()) ---")
display(
    penguins
    .filter("sex.is_not_null()")
    .select('species', 'island', 'body_mass_g', 'sex')
    .head(5)
)

print("\n--- Spaced column using bracket expression ([bill length mm].is_not_null()) ---")
display(
    penguins_spaces
    .filter("[bill length mm].is_not_null()")
    .select('species', 'island', '[bill length mm]', 'sex')
    .head(5)
)

--- Missing sex using string expression (sex.is_null()) ---


species,island,body_mass_g,sex
str,str,f64,str
"""Adelie""","""Torgersen""",null,null
"""Adelie""","""Torgersen""",3475.0,null
"""Adelie""","""Torgersen""",4250.0,null
"""Adelie""","""Torgersen""",3300.0,null
"""Adelie""","""Torgersen""",3700.0,null



--- Known sex using string expression (sex.is_not_null()) ---


species,island,body_mass_g,sex
str,str,f64,str
"""Adelie""","""Torgersen""",3750.0,"""Male"""
"""Adelie""","""Torgersen""",3800.0,"""Female"""
"""Adelie""","""Torgersen""",3250.0,"""Female"""
"""Adelie""","""Torgersen""",3450.0,"""Female"""
"""Adelie""","""Torgersen""",3650.0,"""Male"""



--- Spaced column using bracket expression ([bill length mm].is_not_null()) ---


species,island,bill length mm,sex
str,str,f64,str
"""Adelie""","""Torgersen""",39.1,"""Male"""
"""Adelie""","""Torgersen""",39.5,"""Female"""
"""Adelie""","""Torgersen""",40.3,"""Female"""
"""Adelie""","""Torgersen""",36.7,"""Female"""
"""Adelie""","""Torgersen""",39.3,"""Male"""


In [26]:
# 2. Missing data checks via dictionaries:
print("--- Missing sex using dictionary ({'sex': 'is_null()'}) ---")
display(
    penguins
    .filter({"sex": "is_null()"})
    .select('species', 'island', 'body_mass_g', 'sex')
    .head(5)
)

print("\n--- Known sex using dictionary ({'sex': 'is_not_null()'}) ---")
display(
    penguins
    .filter({"sex": "is_not_null()"})
    .select('species', 'island', 'body_mass_g', 'sex')
    .head(5)
)

print("\n--- Spaced column using dictionary ({'bill length mm': 'is_not_null()'}) ---")
display(
    penguins_spaces
    .filter({"bill length mm": "is_not_null()"})
    .select('species', 'island', '[bill length mm]', 'sex')
    .head(5)
)

--- Missing sex using dictionary ({'sex': 'is_null()'}) ---


species,island,body_mass_g,sex
str,str,f64,str
"""Adelie""","""Torgersen""",null,null
"""Adelie""","""Torgersen""",3475.0,null
"""Adelie""","""Torgersen""",4250.0,null
"""Adelie""","""Torgersen""",3300.0,null
"""Adelie""","""Torgersen""",3700.0,null



--- Known sex using dictionary ({'sex': 'is_not_null()'}) ---


species,island,body_mass_g,sex
str,str,f64,str
"""Adelie""","""Torgersen""",3750.0,"""Male"""
"""Adelie""","""Torgersen""",3800.0,"""Female"""
"""Adelie""","""Torgersen""",3250.0,"""Female"""
"""Adelie""","""Torgersen""",3450.0,"""Female"""
"""Adelie""","""Torgersen""",3650.0,"""Male"""



--- Spaced column using dictionary ({'bill length mm': 'is_not_null()'}) ---


species,island,bill length mm,sex
str,str,f64,str
"""Adelie""","""Torgersen""",39.1,"""Male"""
"""Adelie""","""Torgersen""",39.5,"""Female"""
"""Adelie""","""Torgersen""",40.3,"""Female"""
"""Adelie""","""Torgersen""",36.7,"""Female"""
"""Adelie""","""Torgersen""",39.3,"""Male"""


In [27]:
# 3. Floating-point NaN checks via string expressions and dictionaries:
df_measurements = pt.DataFrame({
    "sample_id": ["S1", "S2", "S3", "S4"],
    "score": [98.5, float("nan"), None, 76.2],
})

print("--- Original DataFrame (has both float NaN and missing null) ---")
display(df_measurements)

print("\n--- Non-dict string expression: score.is_nan() ---")
display(df_measurements.filter("score.is_nan()"))

print("\n--- Dictionary expression: {'score': 'is_nan()'} ---")
display(df_measurements.filter({"score": "is_nan()"}))

print("\n--- Dictionary expression: {'score': 'is_not_nan()'} ---")
display(df_measurements.filter({"score": "is_not_nan()"}))

--- Original DataFrame (has both float NaN and missing null) ---


sample_id,score
str,f64
"""S1""",98.5
"""S2""",NaN
"""S3""",null
"""S4""",76.2



--- Non-dict string expression: score.is_nan() ---


sample_id,score
str,f64
"""S2""",NaN



--- Dictionary expression: {'score': 'is_nan()'} ---


sample_id,score
str,f64
"""S2""",NaN



--- Dictionary expression: {'score': 'is_not_nan()'} ---


sample_id,score
str,f64
"""S1""",98.5
"""S4""",76.2


In [28]:
# 4. Combining null/NaN checks and dynamic query building:
print("--- Combining in string expression ---")
display(
    penguins
    .filter("sex.is_not_null() and body_mass_g > 4500")
    .select('species', 'body_mass_g', 'sex')
    .head(5)
)

print("\n--- Combining in dictionary ---")
display(
    penguins
    .filter({"sex": "is_not_null()", "species": "Gentoo"})
    .select('species', 'body_mass_g', 'sex')
    .head(5)
)

print("\n--- Dynamic programmatic query construction ---")
criteria = {"island": "Biscoe", "species": "Gentoo"}
criteria["sex"] = "is_not_null()"
display(
    penguins
    .filter(criteria)
    .select('species', 'island', 'body_mass_g', 'sex')
    .head(5)
)

--- Combining in string expression ---


species,body_mass_g,sex
str,f64,str
"""Adelie""",4675.0,"""Male"""
"""Adelie""",4650.0,"""Male"""
"""Adelie""",4600.0,"""Male"""
"""Adelie""",4700.0,"""Male"""
"""Adelie""",4725.0,"""Male"""



--- Combining in dictionary ---


species,body_mass_g,sex
str,f64,str
"""Gentoo""",4500.0,"""Female"""
"""Gentoo""",5700.0,"""Male"""
"""Gentoo""",4450.0,"""Female"""
"""Gentoo""",5700.0,"""Male"""
"""Gentoo""",5400.0,"""Male"""



--- Dynamic programmatic query construction ---


species,island,body_mass_g,sex
str,str,f64,str
"""Gentoo""","""Biscoe""",4500.0,"""Female"""
"""Gentoo""","""Biscoe""",5700.0,"""Male"""
"""Gentoo""","""Biscoe""",4450.0,"""Female"""
"""Gentoo""","""Biscoe""",5700.0,"""Male"""
"""Gentoo""","""Biscoe""",5400.0,"""Male"""


## 9. Column-to-Column Comparisons

You can compare two DataFrame columns directly against each other (e.g., detecting limit breaches or verifying ID reconciliations) in both Python and CLI using bare column names without any `col()` wrapper:

In [29]:
# Comparing two columns directly using bare column names
df_risk = pt.DataFrame({
    "exposure": [120, 250, 80, 410],
    "limit": [100, 200, 150, 400],
    "tranche_id": ["T1", "T2", "T3", "T4"],
    "deal_id": ["T1", "D2", "T3", "D4"],
})

print("--- Breaches (exposure > limit) ---")
display(df_risk.filter("exposure > limit"))

print("\n--- Reconciled (tranche_id == deal_id) ---")
display(df_risk.filter("tranche_id == deal_id"))

--- Breaches (exposure > limit) ---


exposure,limit,tranche_id,deal_id
i64,i64,str,str
120,100,"""T1""","""T1"""
250,200,"""T2""","""D2"""
410,400,"""T4""","""D4"""



--- Reconciled (tranche_id == deal_id) ---


exposure,limit,tranche_id,deal_id
i64,i64,str,str
120,100,"""T1""","""T1"""
80,150,"""T3""","""T3"""


## 10. External Variable Injection (`@var` and `col(@var)`)

You can inject external local variables into filter expressions:
- `@var`: Injects a scalar threshold or comparison literal
- `col(@var)`: Injects a dynamic column name from a Python variable

In [30]:
# Inject scalar threshold and dynamic column target
threshold = 100
target_col = "deal_id"

display(df_risk.filter("exposure > @threshold and tranche_id == col(@target_col)"))

exposure,limit,tranche_id,deal_id
i64,i64,str,str
120,100,"""T1""","""T1"""


## 11. Strict Error Handling & Diagnostic Feedback

`pytae` is intentionally designed with helpful, unambiguous error messages that guide you to the right syntax rather than silently failing or doing unexpected things.

In [31]:
# 1. Keyword arguments are strictly rejected (forces '==' and prevents '=' confusion)
try:
    penguins.filter(species="Gentoo")
except TypeError as err:
    print("Caught expected TypeError:\n", err)

Caught expected TypeError:
 filter(): keyword arguments are not supported (received ['species']). Equality comparisons strictly require '==' to avoid '=' assignment ambiguity. Use a string expression (e.g. df.filter("species == 'Gentoo'")) or a dictionary (e.g. df.filter({'species': 'Gentoo'}))


In [32]:
# 2. Single '=' in strings is strictly disallowed (reserved for mutate)
try:
    penguins.filter("species = 'Gentoo'")
except ValueError as err:
    print("Caught expected ValueError:\n", err)

Caught expected ValueError:
 invalid filter condition 'species = 'Gentoo'': use '==' for equality comparisons (e.g. species == 'Gentoo'). Single '=' is for assignment (-mutate), not comparison (-filter).


In [33]:
# 3. Comma ',' is disallowed as a boolean operator (use 'and')
try:
    penguins.filter("species == 'Gentoo', body_mass_g > 5000")
except ValueError as err:
    print("Caught expected ValueError:\n", err)

Caught expected ValueError:
 invalid filter condition 'species == 'Gentoo', body_mass_g > 5000': comma ',' is not supported as a boolean operator. Use 'and' instead (e.g. species == 'Gentoo' and body_mass_g > 5000).


In [34]:
# 4. Bitwise '&' and '|' are disallowed (use 'and' / 'or')
try:
    penguins.filter("species == 'Gentoo' & body_mass_g > 5000")
except ValueError as err:
    print("Caught expected ValueError:\n", err)

Caught expected ValueError:
 invalid filter condition 'species == 'Gentoo' & body_mass_g > 5000': '&' is not supported in filter. Use 'and' instead.


In [35]:
# 5. Legacy interval strings '[3000, 4000]' are disallowed
try:
    penguins.filter("body_mass_g = [3000, 4000]")
except ValueError as err:
    print("Caught expected ValueError:\n", err)

Caught expected ValueError:
 invalid filter condition 'body_mass_g = [3000, 4000]': use '==' for equality comparisons (e.g. body_mass_g == [3000, 4000]). Single '=' is for assignment (-mutate), not comparison (-filter).


In [36]:
# 6. String column type safety: unquoted int against string column raises TypeError
df_str = pt.DataFrame({"id": ["1", "9", "10"], "val": [100, 200, 300]})

try:
    df_str.filter("id == 9")
except TypeError as err:
    print("Caught expected TypeError:\n", err)

Caught expected TypeError:
 filter(): column 'id' has String type, but received int 9. String values must be enclosed in quotes (e.g. id = '9').


In [37]:
# 7. Typo suggestion on misspelled column names
try:
    penguins.filter("speceis == 'Adelie'")
except (KeyError, Exception) as err:
    print("Caught expected error:\n", err)

Caught expected error:
 unable to find column "speceis"; valid columns: ["species", "island", "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex"]

Did you mean "species"?


## 12. Real-World Analytical Pipelines

Filter works seamlessly as the entry point or intermediate step in pytae pipelines combining filtering, grouping, aggregation, reshaping, and sorting.

In [38]:
# Tips dataset: weekend dinner parties of 2+ spending > $15
(
    tips
    .filter("day in ['Sat', 'Sun'] and time == 'Dinner' and size >= 2 and total_bill > 15")
    .by('smoker')
    .agg("mean_tip = tip:mean, mean_bill = total_bill:mean, total_diners = size:sum, n_tables = count")
    .arrange("mean_tip desc")
)

smoker,mean_tip,mean_bill,total_diners,n_tables
cat,f64,f64,i64,i64
"""No""",3.521081,23.257297,225,74
"""Yes""",3.384889,26.112,122,45


In [39]:
# Titanic dataset: survival rate of passengers in 1st/2nd class by sex
(
    titanic
    .filter("pclass in [1, 2] and age.is_not_null()")
    .filter({"age": ("between", 18, 60)})
    .by('pclass', 'sex')
    .agg("survival_rate = survived:mean, n_passengers = count")
    .arrange("pclass, sex")
)

pclass,sex,survival_rate,n_passengers
i64,str,f64,i64
1,"""female""",0.973333,75
1,"""male""",0.411765,85
2,"""female""",0.903226,62
2,"""male""",0.058824,85
